# Percepción de inseguridad en Quito, 2023–2025
## Difference-in-Differences por Administración Zonal

Este notebook reproduce el análisis de cambio relativo de la percepción de inseguridad entre 2023 y 2025.

Para cada Administración Zonal:

**DiD = (cambio 2023→2025 de la zona) − (cambio 2023→2025 del resto de Quito)**

- `DiD < 0`: mejora relativa.
- `DiD > 0`: deterioro relativo.

Se utiliza el factor de expansión individual `peso_az_estrato`.

**Nota:** con solo dos períodos no se pueden evaluar tendencias paralelas previas; por eso los resultados se interpretan principalmente como comparaciones de cambio relativo.


In [ ]:
!pip -q install openpyxl statsmodels seaborn


In [ ]:
import os
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from statsmodels.stats.multitest import multipletests
from google.colab import files

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid")


## 1. Cargar la base


In [ ]:
FILE_NAME = "EVPI_2024_2026_VF_1.xlsx"

if not os.path.exists(FILE_NAME):
    print(f"No se encontró {FILE_NAME}. Selecciónalo desde tu equipo.")
    uploaded = files.upload()
    FILE_NAME = next(iter(uploaded.keys()))

df = pd.read_excel(FILE_NAME, sheet_name="Sheet1", engine="openpyxl")

print("Dimensiones de la base:", df.shape)
print("Archivo:", FILE_NAME)


## 2. Variables principales y selección de indicadores comparables


In [ ]:
YEAR = "AÑO"
ZONE = "ADMINISTRACION ZONAL"
WEIGHT = "peso_az_estrato"

required = [YEAR, ZONE, WEIGHT]
missing = [x for x in required if x not in df.columns]
assert not missing, f"Faltan columnas requeridas: {missing}"

def find_col(prefix):
    matches = [c for c in df.columns if str(c).strip().startswith(prefix)]
    if len(matches) != 1:
        raise ValueError(f"Para {prefix} se encontraron {len(matches)} columnas: {matches}")
    return matches[0]

psc_nums = [1, 2, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15]
psc_cols = {n: find_col(f"PSC_{n} -") for n in psc_nums}

labels = {
    1: "Casa/cuadra",
    2: "Barrio",
    4: "Centros educativos",
    5: "Mercados/ferias",
    6: "Cajeros vía pública",
    7: "Espacios públicos",
    8: "Metro",
    9: "Trole/Ecovía/corredores",
    11: "Taxis",
    12: "Apps de transporte",
    13: "Espectáculos públicos",
    14: "Caminar por el barrio de día",
    15: "Caminar por el barrio de noche",
}

print("Años disponibles:", sorted(pd.to_numeric(df[YEAR], errors="coerce").dropna().unique()))
print("\nVariables PSC utilizadas:")
for n, c in psc_cols.items():
    print(f"PSC_{n}: {c}")


## 3. Limpieza de la muestra y ponderadores


In [ ]:
d = df.copy()

d[YEAR] = pd.to_numeric(d[YEAR], errors="coerce")
d[WEIGHT] = pd.to_numeric(d[WEIGHT], errors="coerce")

d = d[d[YEAR].isin([2023, 2025])].copy()
d = d[d[ZONE].notna() & d[WEIGHT].notna() & (d[WEIGHT] > 0)].copy()
d["post"] = (d[YEAR] == 2025).astype(int)

print("N por año:")
display(d.groupby(YEAR).size().rename("N").to_frame())

print("Suma de factores de expansión:")
display(d.groupby(YEAR)[WEIGHT].sum().rename("Población expandida").to_frame())


## 4. Recodificación de inseguridad

Se codifica:

- `1` = Muy inseguro + inseguro
- `0` = Muy seguro + seguro
- `NaN` = No aplica / otra respuesta


In [ ]:
def recode_insecurity(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s.startswith("1."):
        return 1.0
    if s.startswith("2."):
        return 0.0
    return np.nan

for n, col in psc_cols.items():
    d[f"ins_{n}"] = d[col].map(recode_insecurity)

validity = pd.DataFrame({
    "Variable": [labels[n] for n in psc_nums],
    "N_válidos": [d[f"ins_{n}"].notna().sum() for n in psc_nums],
    "%_válidos": [100*d[f"ins_{n}"].notna().mean() for n in psc_nums],
})
display(validity.round(2))


## 5. Índices sintéticos


In [ ]:
barrio_vars = [f"ins_{n}" for n in [1, 2, 14, 15]]
public_vars = [f"ins_{n}" for n in [4, 5, 6, 7, 8, 9, 11, 12, 13]]
general_vars = [f"ins_{n}" for n in psc_nums]

d["idx_barrial"] = d[barrio_vars].mean(axis=1, skipna=True)
d.loc[d[barrio_vars].notna().sum(axis=1) < 2, "idx_barrial"] = np.nan

d["idx_publico_transporte"] = d[public_vars].mean(axis=1, skipna=True)
d.loc[d[public_vars].notna().sum(axis=1) < 5, "idx_publico_transporte"] = np.nan

d["idx_general"] = d[general_vars].mean(axis=1, skipna=True)
d.loc[d[general_vars].notna().sum(axis=1) < 8, "idx_general"] = np.nan


## 6. Funciones de medias ponderadas y DiD


In [ ]:
def wmean(data, outcome, weight=WEIGHT):
    tmp = data[[outcome, weight]].dropna()
    if tmp.empty:
        return np.nan
    return np.average(tmp[outcome], weights=tmp[weight])

def did_zone(data, outcome, zone_name):
    tmp = data[[outcome, ZONE, YEAR, WEIGHT, "post"]].dropna().copy()
    tmp["zona"] = (tmp[ZONE].astype(str) == str(zone_name)).astype(int)
    tmp["interaction"] = tmp["zona"] * tmp["post"]

    X = sm.add_constant(tmp[["zona", "post", "interaction"]], has_constant="add")
    model = sm.WLS(tmp[outcome], X, weights=tmp[WEIGHT]).fit(cov_type="HC1")

    z23 = tmp[(tmp["zona"] == 1) & (tmp[YEAR] == 2023)]
    z25 = tmp[(tmp["zona"] == 1) & (tmp[YEAR] == 2025)]
    r23 = tmp[(tmp["zona"] == 0) & (tmp[YEAR] == 2023)]
    r25 = tmp[(tmp["zona"] == 0) & (tmp[YEAR] == 2025)]

    mean_z23 = wmean(z23, outcome)
    mean_z25 = wmean(z25, outcome)
    mean_r23 = wmean(r23, outcome)
    mean_r25 = wmean(r25, outcome)

    coef = model.params["interaction"]
    se = model.bse["interaction"]
    p = model.pvalues["interaction"]

    return {
        "zona": zone_name,
        "outcome": outcome,
        "zona_2023": mean_z23,
        "zona_2025": mean_z25,
        "cambio_zona": mean_z25 - mean_z23,
        "resto_2023": mean_r23,
        "resto_2025": mean_r25,
        "cambio_resto": mean_r25 - mean_r23,
        "did": coef,
        "se": se,
        "ci_low": coef - 1.96*se,
        "ci_high": coef + 1.96*se,
        "p": p,
        "n": len(tmp),
    }

zones = sorted(d[ZONE].dropna().astype(str).unique())
print("Zonas:", zones)


## 7. Evolución de Quito en los 13 indicadores


In [ ]:
city_rows = []

for n in psc_nums:
    y = f"ins_{n}"
    m23 = wmean(d[d[YEAR] == 2023], y)
    m25 = wmean(d[d[YEAR] == 2025], y)
    city_rows.append({
        "Indicador": labels[n],
        "2023": 100*m23,
        "2025": 100*m25,
        "Cambio_pp": 100*(m25-m23),
    })

city = pd.DataFrame(city_rows)
display(city.round(2))

plot_city = city.sort_values("Cambio_pp")
plt.figure(figsize=(9, 6))
plt.barh(plot_city["Indicador"], plot_city["Cambio_pp"])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Cambio 2023–2025 (puntos porcentuales)")
plt.ylabel("")
plt.title("Cambio en percepción de inseguridad en Quito")
plt.tight_layout()
plt.show()


## 8. Evolución de los índices en toda la ciudad


In [ ]:
index_labels = {
    "idx_barrial": "Índice barrial",
    "idx_publico_transporte": "Espacios públicos y transporte",
    "idx_general": "Índice general (13 indicadores)",
}

index_city = []
for outcome, label in index_labels.items():
    m23 = wmean(d[d[YEAR] == 2023], outcome)
    m25 = wmean(d[d[YEAR] == 2025], outcome)
    index_city.append({
        "Índice": label,
        "2023": 100*m23,
        "2025": 100*m25,
        "Cambio_pp": 100*(m25-m23),
    })

index_city = pd.DataFrame(index_city)
display(index_city.round(2))

print(
    "Chequeo orientativo: con la misma versión de la base, "
    "el índice general debería quedar aproximadamente en 66.9% en 2023 "
    "y 61.7% en 2025."
)


## 9. DiD del índice general por Administración Zonal


In [ ]:
results_general = pd.DataFrame(
    [did_zone(d, "idx_general", z) for z in zones]
)

results_general["q_fdr"] = multipletests(
    results_general["p"].fillna(1),
    alpha=0.05,
    method="fdr_bh"
)[1]

for c in [
    "zona_2023", "zona_2025", "cambio_zona",
    "resto_2023", "resto_2025", "cambio_resto",
    "did", "se", "ci_low", "ci_high"
]:
    results_general[c + "_pp"] = 100*results_general[c]

table_general = results_general[
    [
        "zona", "zona_2023_pp", "zona_2025_pp", "cambio_zona_pp",
        "cambio_resto_pp", "did_pp", "ci_low_pp", "ci_high_pp",
        "p", "q_fdr"
    ]
].sort_values("did_pp")

display(table_general.round(3))


In [ ]:
plot = table_general.sort_values("did_pp").copy()
ypos = np.arange(len(plot))

plt.figure(figsize=(9, 6))
plt.errorbar(
    plot["did_pp"],
    ypos,
    xerr=[
        plot["did_pp"] - plot["ci_low_pp"],
        plot["ci_high_pp"] - plot["did_pp"]
    ],
    fmt="o",
    capsize=3
)
plt.axvline(0, color="black", linewidth=0.9)
plt.yticks(ypos, plot["zona"])
plt.xlabel("DiD (puntos porcentuales)")
plt.title("Índice general de inseguridad: DiD por Administración Zonal")
plt.tight_layout()
plt.show()


## 10. DiD para los 13 indicadores individuales


In [ ]:
all_results = []

for n in psc_nums:
    outcome = f"ins_{n}"
    tmp = pd.DataFrame([did_zone(d, outcome, z) for z in zones])
    tmp["indicador"] = labels[n]
    tmp["psc"] = f"PSC_{n}"
    tmp["q_fdr"] = multipletests(
        tmp["p"].fillna(1),
        alpha=0.05,
        method="fdr_bh"
    )[1]
    all_results.append(tmp)

results_indicators = pd.concat(all_results, ignore_index=True)
results_indicators["did_pp"] = 100*results_indicators["did"]
results_indicators["ci_low_pp"] = 100*results_indicators["ci_low"]
results_indicators["ci_high_pp"] = 100*results_indicators["ci_high"]

display(
    results_indicators[
        ["zona", "indicador", "did_pp", "ci_low_pp", "ci_high_pp", "p", "q_fdr"]
    ].sort_values(["zona", "did_pp"]).round(3)
)


In [ ]:
heat = results_indicators.pivot(
    index="zona",
    columns="indicador",
    values="did_pp"
)

zone_order = table_general["zona"].tolist()
heat = heat.reindex(zone_order)

plt.figure(figsize=(15, 7))
sns.heatmap(
    heat,
    center=0,
    annot=True,
    fmt=".1f",
    linewidths=.5,
    cbar_kws={"label": "DiD (pp)"}
)
plt.title("Cambio relativo 2023–2025: cada zona vs. resto de Quito")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 11. Robustez: percepción de que la inseguridad aumentó


In [ ]:
change_nums = [16, 17, 18]
change_labels = {
    16: "Aumentó inseguridad en el barrio",
    17: "Aumentó inseguridad en Quito",
    18: "Aumentó inseguridad en el país",
}

change_cols = {n: find_col(f"PSC_{n} -") for n in change_nums}

def recode_increased(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s.startswith("1.") and "aument" in s:
        return 1.0
    if s.startswith("2.") or s.startswith("3."):
        return 0.0
    return np.nan

for n, col in change_cols.items():
    d[f"increase_{n}"] = d[col].map(recode_increased)

robust_rows = []
for n in change_nums:
    outcome = f"increase_{n}"
    tmp = pd.DataFrame([did_zone(d, outcome, z) for z in zones])
    tmp["indicador"] = change_labels[n]
    tmp["q_fdr"] = multipletests(tmp["p"].fillna(1), method="fdr_bh")[1]
    tmp["did_pp"] = 100*tmp["did"]
    robust_rows.append(tmp)

robust = pd.concat(robust_rows, ignore_index=True)

display(
    robust[["zona", "indicador", "did_pp", "p", "q_fdr"]]
    .sort_values(["indicador", "did_pp"])
    .round(3)
)


## 12. Exportar resultados


In [ ]:
OUTPUT_DIR = Path("resultados_did_quito")
OUTPUT_DIR.mkdir(exist_ok=True)

city.to_csv(OUTPUT_DIR / "01_cambio_indicadores_quito.csv", index=False)
index_city.to_csv(OUTPUT_DIR / "02_indices_quito.csv", index=False)
table_general.to_csv(OUTPUT_DIR / "03_did_indice_general_por_zona.csv", index=False)
results_indicators.to_csv(OUTPUT_DIR / "04_did_13_indicadores_por_zona.csv", index=False)
robust.to_csv(OUTPUT_DIR / "05_robustez_percepcion_aumento.csv", index=False)

zip_path = "resultados_did_quito.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUTPUT_DIR.glob("*.csv"):
        z.write(f, arcname=f.name)

print("ZIP generado:", zip_path)
files.download(zip_path)


## 13. Interpretación metodológica

Este diseño permite comparar cómo cambia cada Administración Zonal respecto del resto de la ciudad.

No obstante, con solo 2023 y 2025 no se puede observar una tendencia pretratamiento. Por ello, el coeficiente DiD debe leerse como **cambio relativo** y no automáticamente como un efecto causal de una política.

Para una versión inferencial definitiva, si la base contiene identificadores de UPM/PSU, conglomerados y estratos del diseño muestral, conviene incorporarlos en la estimación de errores estándar.
